In [ ]:
import os
from typing import Optional

import pyodbc

# SQL Server connection

server = os.getenv("SQL_SERVER", r"************")
database = os.getenv("SQL_DATABASE", "sales")
driver = os.getenv("SQL_DRIVER", "ODBC Driver 18 for SQL Server")
encrypt = os.getenv("SQL_ENCRYPT", "yes")
trust_cert = os.getenv("SQL_TRUST_SERVER_CERT", "yes")
username = os.getenv("SQL_USER")
password = os.getenv("SQL_PASSWORD")

config = {
    "server": server,
    "database": database,
    "driver": driver,
    "encrypt": encrypt,
    "trust_cert": trust_cert,
    "username": "***" if username else None,
}

config

def build_connection_string(
    server: str,
    database: str,
    driver: str,
    encrypt: str = "yes",
    trust_server_certificate: str = "yes",
    username: Optional[str] = None,
    password: Optional[str] = None,
) -> str:
    if not server:
        raise ValueError("server is required")
    if not database:
        raise ValueError("database is required")
    if not driver:
        raise ValueError("driver is required")

    parts = [
        f"DRIVER={{{driver}}}",
        f"SERVER={server}",
        f"DATABASE={database}",
        f"Encrypt={encrypt}",
        f"TrustServerCertificate={trust_server_certificate}",
    ]

    if username and password:
        parts.append(f"UID={username}")
        parts.append(f"PWD={password}")
    else:
        parts.append("Trusted_Connection=yes")

    return ";".join(parts)


def create_connection(conn_str: str, timeout_seconds: int = 15) -> pyodbc.Connection:
    return pyodbc.connect(conn_str, timeout=timeout_seconds)


conn_str = build_connection_string(
    server=server,
    database=database,
    driver=driver,
    encrypt=encrypt,
    trust_server_certificate=trust_cert,
    username=username,
    password=password,
)

conn = create_connection(conn_str)

with conn.cursor() as cursor:
    cursor.execute("SELECT 1 AS health_check")
    print("Health check:", cursor.fetchone()[0])

print("Connection opened successfully.")

Health check: 1
Connection opened successfully.


In [ ]:
# Load raw data to bronze

import pandas as pd
import json

def read_source_file(source_type: str, source_path: str) -> pd.DataFrame:
    """Read a source file based on its type and return a DataFrame."""
    source_type = source_type.lower()
    if source_type == "xlsx":
        return pd.read_excel(source_path)
    elif source_type == "csv":
        return pd.read_csv(source_path)
    elif source_type == "json":
        return pd.read_json(source_path)
    else:
        raise ValueError(f"Unsupported source type: {source_type}")


def sanitize_columns(df: pd.DataFrame) -> pd.DataFrame:
    """Sanitize column names to be SQL-safe."""
    df.columns = [
        col.strip().replace(" ", "_").replace("-", "_").replace(".", "_")
        for col in df.columns
    ]
    return df


def create_bronze_table(cursor: pyodbc.Cursor, table_name: str, df: pd.DataFrame) -> None:
    """Drop and recreate a bronze table based on DataFrame schema."""
    type_map = {
        "int64": "BIGINT",
        "float64": "FLOAT",
        "bool": "BIT",
        "datetime64[ns]": "DATETIME2",
        "object": "NVARCHAR(MAX)",
    }

    columns_ddl = ",\n    ".join(
        f"[{col}] {type_map.get(str(dtype), 'NVARCHAR(MAX)')}"
        for col, dtype in zip(df.columns, df.dtypes)
    )

    cursor.execute(f"IF OBJECT_ID('bronze.{table_name}', 'U') IS NOT NULL DROP TABLE bronze.{table_name};")
    cursor.execute(f"CREATE TABLE bronze.{table_name} (\n    {columns_ddl}\n);")


def insert_into_bronze(cursor: pyodbc.Cursor, table_name: str, df: pd.DataFrame) -> None:
    """Insert DataFrame rows into a bronze table."""
    placeholders = ", ".join(["?" for _ in df.columns])
    columns = ", ".join([f"[{col}]" for col in df.columns])
    sql = f"INSERT INTO bronze.{table_name} ({columns}) VALUES ({placeholders})"
    rows_to_insert = [
        tuple(None if pd.isna(v) else v for v in row)
        for row in df.itertuples(index=False, name=None)
    ]
    cursor.executemany(sql, rows_to_insert)


def load_sources_to_bronze(connection: pyodbc.Connection) -> None:
    """Read all sources from meta_source and load them into bronze schema."""
    with connection.cursor() as cur:
        cur.execute("SELECT source_name, source_type, source_path FROM sales.dbo.meta_source")
        sources = cur.fetchall()

    for source_name, source_type, source_path in sources:
        print(f"Processing: {source_name} ({source_type}) from {source_path}")
        try:
            df = read_source_file(source_type, source_path)
            df = sanitize_columns(df)

            with connection.cursor() as cur:
                create_bronze_table(cur, source_name, df)
                insert_into_bronze(cur, source_name, df)

            connection.commit()
            print(f"  -> Loaded {len(df)} rows into bronze.{source_name}")
        except Exception as e:
            connection.rollback()
            print(f"  -> ERROR loading {source_name}: {e}")


# Run the ingestion
load_sources_to_bronze(conn)

Processing: customer (xlsx) from c:\Users\JAD3KOR\OneDrive - Bosch Group\Newfolder\Guide\New folder\New folder (2)\New folder (2)\Lead_Data_Engineer_Assessment\Dataset\customer.xlsx
  -> Loaded 793 rows into bronze.customer
Processing: orders (json) from c:\Users\JAD3KOR\OneDrive - Bosch Group\Newfolder\Guide\New folder\New folder (2)\New folder (2)\Lead_Data_Engineer_Assessment\Dataset\orders.json
  -> Loaded 9994 rows into bronze.orders
Processing: products (csv) from c:\Users\JAD3KOR\OneDrive - Bosch Group\Newfolder\Guide\New folder\New folder (2)\New folder (2)\Lead_Data_Engineer_Assessment\Dataset\products.csv
  -> Loaded 1851 rows into bronze.products


In [ ]:
# Load enrich bronze and load to silver

def ensure_schema(connection: pyodbc.Connection, schema_name: str = "silver") -> None:
    with connection.cursor() as cur:
        cur.execute(f"""
        IF NOT EXISTS (SELECT 1 FROM sys.schemas WHERE name = '{schema_name}')
        BEGIN
            EXEC('CREATE SCHEMA {schema_name} AUTHORIZATION dbo');
        END
        """)
    connection.commit()


def read_sql_table(connection: pyodbc.Connection, schema: str, table: str) -> pd.DataFrame:
    return pd.read_sql_query(f"SELECT * FROM {schema}.{table}", connection)


def deduplicate_dataframe(df: pd.DataFrame, key_columns: list[str] | None = None) -> pd.DataFrame:
    if not key_columns:
        return df.drop_duplicates().reset_index(drop=True)

    valid_keys = [c for c in key_columns if c in df.columns]
    if not valid_keys:
        print(f"  -> Dedup keys {key_columns} not found. Falling back to full-row dedup.")
        return df.drop_duplicates().reset_index(drop=True)

    return df.drop_duplicates(subset=valid_keys, keep="first").reset_index(drop=True)


def create_table_from_df(
    cursor: pyodbc.Cursor,
    schema: str,
    table: str,
    df: pd.DataFrame
) -> None:
    type_map = {
        "int64": "BIGINT",
        "float64": "FLOAT",
        "bool": "BIT",
        "datetime64[ns]": "DATETIME2",
        "object": "NVARCHAR(MAX)",
    }

    columns_ddl = ",\n    ".join(
        f"[{col}] {type_map.get(str(dtype), 'NVARCHAR(MAX)')}"
        for col, dtype in zip(df.columns, df.dtypes)
    )

    cursor.execute(f"IF OBJECT_ID('{schema}.{table}', 'U') IS NOT NULL DROP TABLE {schema}.{table};")
    cursor.execute(f"CREATE TABLE {schema}.{table} (\n    {columns_ddl}\n);")


def insert_dataframe(cursor: pyodbc.Cursor, schema: str, table: str, df: pd.DataFrame) -> None:
    if df.empty:
        return

    placeholders = ", ".join(["?" for _ in df.columns])
    columns = ", ".join([f"[{c}]" for c in df.columns])
    sql = f"INSERT INTO {schema}.{table} ({columns}) VALUES ({placeholders})"

    rows_to_insert = [
        tuple(None if pd.isna(v) else v for v in row)
        for row in df.itertuples(index=False, name=None)
    ]
    cursor.executemany(sql, rows_to_insert)


def load_bronze_to_silver(
    connection: pyodbc.Connection,
    table_name: str,
    dedup_keys: list[str] | None = None
) -> None:
    print(f"Processing bronze.{table_name} -> silver.{table_name}")
    try:
        src_df = read_sql_table(connection, "bronze", table_name)
        dedup_df = deduplicate_dataframe(src_df, dedup_keys)

        with connection.cursor() as cur:
            create_table_from_df(cur, "silver", table_name, dedup_df)
            insert_dataframe(cur, "silver", table_name, dedup_df)

        connection.commit()
        print(f"  -> Loaded {len(dedup_df)} rows (from {len(src_df)} source rows)")
    except Exception as e:
        connection.rollback()
        print(f"  -> ERROR: {e}")
        raise


# Run for customer and products
ensure_schema(conn, "silver")

silver_load_plan = {
    "customer": ["Customer_ID", "customer_id"],
    "products": ["Product_ID", "product_id"],
}

for table, keys in silver_load_plan.items():
    load_bronze_to_silver(conn, table, keys)


def load_enriched_orders_to_silver(connection: pyodbc.Connection) -> None:
    print("Processing bronze.orders -> silver.orders")

    try:
        orders_df = read_sql_table(connection, "bronze", "orders")
        customer_df = read_sql_table(connection, "silver", "customer")
        products_df = read_sql_table(connection, "silver", "products")

        orders_customer_key = "Customer_ID"
        orders_product_key = "Product_ID"
        orders_profit_col = "Profit"

        customer_key = "Customer_ID"
        customer_name_col = "Customer_Name"
        customer_country_col = "Country"

        product_key = "Product_ID"
        product_category_col = "Category"
        product_sub_category_col = "Sub_Category"
       

        enriched_df = orders_df.merge(
            customer_df[[customer_key, customer_name_col, customer_country_col]],
            left_on=orders_customer_key,
            right_on=customer_key,
            how="left"
        ).merge(
            products_df[[product_key, product_category_col, product_sub_category_col]],
            left_on=orders_product_key,
            right_on=product_key,
            how="left"
        )

        enriched_df = enriched_df.loc[:, ~enriched_df.columns.duplicated()]

        enriched_df[orders_profit_col] = pd.to_numeric(
            enriched_df[orders_profit_col], errors="coerce"
        ).round(2)

        enriched_df = deduplicate_dataframe(enriched_df)

        with connection.cursor() as cur:
            create_table_from_df(cur, "silver", "orders", enriched_df)
            insert_dataframe(cur, "silver", "orders", enriched_df)

        connection.commit()
        print(f"  -> Loaded {len(enriched_df)} rows into silver.orders")
    except Exception as e:
        connection.rollback()
        print(f"  -> ERROR: {e}")
        raise


load_enriched_orders_to_silver(conn)

Processing bronze.customer -> silver.customer


C:\Users\JAD3KOR\AppData\Local\Temp\ipykernel_35892\3312688934.py:15: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql_query(f"SELECT * FROM {schema}.{table}", connection)


  -> Loaded 793 rows (from 793 source rows)
Processing bronze.products -> silver.products


C:\Users\JAD3KOR\AppData\Local\Temp\ipykernel_35892\3312688934.py:15: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql_query(f"SELECT * FROM {schema}.{table}", connection)


  -> Loaded 1818 rows (from 1851 source rows)
Processing bronze.orders -> silver.orders


C:\Users\JAD3KOR\AppData\Local\Temp\ipykernel_35892\3312688934.py:15: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql_query(f"SELECT * FROM {schema}.{table}", connection)


  -> Loaded 9994 rows into silver.orders


In [ ]:
# Aggregate and load to gold

ensure_schema(conn, "gold")

orders_gold_df = read_sql_table(conn, "silver", "orders")

date_col = "Order_Date"
category_col = "Category"
sub_category_col = "Sub_Category"
customer_col = "Customer_Name"
profit_col = "Profit"


orders_gold_df[date_col] = pd.to_datetime(orders_gold_df[date_col], errors="coerce")
orders_gold_df[profit_col] = pd.to_numeric(orders_gold_df[profit_col], errors="coerce")

agg_profit_df = (
    orders_gold_df.dropna(subset=[date_col])
    .assign(Year=orders_gold_df[date_col].dt.year)
    .groupby(
        ["Year", category_col, sub_category_col, customer_col],
        dropna=False,
        as_index=False
    )[profit_col]
    .sum()
    .rename(
        columns={
            category_col: "Product_Category",
            sub_category_col: "Product_Sub_Category",
            customer_col: "Customer",
            profit_col: "Total_Profit",
        }
    )
    .sort_values(
        ["Year", "Product_Category", "Product_Sub_Category", "Customer"]
    )
    .reset_index(drop=True)
)

with conn.cursor() as cur:
    create_table_from_df(cur, "gold", "agg_profit_by_year_category_subcategory_customer", agg_profit_df)
    insert_dataframe(cur, "gold", "agg_profit_by_year_category_subcategory_customer", agg_profit_df)

conn.commit()
print(f"Loaded {len(agg_profit_df)} rows into gold.agg_profit_by_year_category_subcategory_customer")

agg_profit_df.head()

C:\Users\JAD3KOR\AppData\Local\Temp\ipykernel_35892\3312688934.py:15: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql_query(f"SELECT * FROM {schema}.{table}", connection)
C:\Users\JAD3KOR\AppData\Local\Temp\ipykernel_35892\3298106461.py:33: UserWarning: Parsing dates in %d/%m/%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  orders_gold_df[date_col] = pd.to_datetime(orders_gold_df[date_col], errors="coerce")


Loaded 8073 rows into gold.agg_profit_by_year_category_subcategory_customer


,Year,Product_Category,Product_Sub_Category,Customer,Total_Profit
0,2014,Furniture,Bookcases,Art Foster,-216.98
1,2014,Furniture,Bookcases,Bobby Trafton,-44.20
2,2014,Furniture,Bookcases,Brendan Sweed,-53.29
3,2014,Furniture,Bookcases,Brian Dahlen,3.93
4,2014,Furniture,Bookcases,Brian Moss,20.52


In [15]:
# Aggregates from gold layer using SQL

base_table = "gold.agg_profit_by_year_category_subcategory_customer"

sql_profit_by_year = f"""
SELECT
    [Year],
    SUM([Total_Profit]) AS Profit_By_Year
FROM {base_table}
GROUP BY [Year]
ORDER BY [Year];
"""

sql_profit_by_year_category = f"""
SELECT
    [Year],
    COALESCE([Product_Category], 'Unknown') AS Product_Category,
    SUM([Total_Profit]) AS Profit_By_Year_Category
FROM {base_table}
GROUP BY [Year], COALESCE([Product_Category], 'Unknown')
ORDER BY [Year], Product_Category;
"""

sql_profit_by_customer = f"""
SELECT
    COALESCE([Customer], 'Unknown') AS Customer,
    SUM([Total_Profit]) AS Profit_By_Customer
FROM {base_table}
GROUP BY COALESCE([Customer], 'Unknown')
ORDER BY Profit_By_Customer DESC;
"""

sql_profit_by_customer_year = f"""
SELECT
    COALESCE([Customer], 'Unknown') AS Customer,
    [Year],
    SUM([Total_Profit]) AS Profit_By_Customer_Year
FROM {base_table}
GROUP BY COALESCE([Customer], 'Unknown'), [Year]
ORDER BY Customer, [Year];
"""

profit_by_year_df = pd.read_sql_query(sql_profit_by_year, conn)
profit_by_year_category_df = pd.read_sql_query(sql_profit_by_year_category, conn)
profit_by_customer_df = pd.read_sql_query(sql_profit_by_customer, conn)
profit_by_customer_year_df = pd.read_sql_query(sql_profit_by_customer_year, conn)

print("Profit by Year")
display(profit_by_year_df)

print("Profit by Year + Product Category")
display(profit_by_year_category_df)

print("Profit by Customer")
display(profit_by_customer_df)

print("Profit by Customer + Year")
display(profit_by_customer_year_df)

C:\Users\JAD3KOR\AppData\Local\Temp\ipykernel_35892\2783733479.py:43: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  profit_by_year_df = pd.read_sql_query(sql_profit_by_year, conn)
C:\Users\JAD3KOR\AppData\Local\Temp\ipykernel_35892\2783733479.py:44: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  profit_by_year_category_df = pd.read_sql_query(sql_profit_by_year_category, conn)
C:\Users\JAD3KOR\AppData\Local\Temp\ipykernel_35892\2783733479.py:45: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  profit_by_customer_df = pd.read_sql

Profit by Year


C:\Users\JAD3KOR\AppData\Local\Temp\ipykernel_35892\2783733479.py:46: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  profit_by_customer_year_df = pd.read_sql_query(sql_profit_by_customer_year, conn)


,Year,Profit_By_Year
0,2014,39185.75
1,2015,63073.09
2,2016,65073.23
3,2017,111084.96


Profit by Year + Product Category


,Year,Product_Category,Profit_By_Year_Category
0,2014,Furniture,-5331.04
1,2014,Office Supplies,22500.32
2,2014,Technology,21493.35
3,2014,Unknown,523.12
4,2015,Furniture,3027.18
5,2015,Office Supplies,24519.39
6,2015,Technology,34943.36
7,2015,Unknown,583.16
8,2016,Furniture,6889.50
9,2016,Office Supplies,34555.53


Profit by Customer


,Customer,Profit_By_Customer
0,Frank Hawley,13400.32
1,Tamara Chand,8981.32
2,Raymond Buch,6976.36
3,Sanjit Chand,5757.30
4,Hunter Lopez,5622.43
...,...,...
781,Luke Foster,-3583.61
782,Grant Thornton,-4108.66
783,Cindy Stewart,-6626.24
784,Ed Jacobs,-6964.03


Profit by Customer + Year


,Customer,Year,Profit_By_Customer_Year
0,Dorothy Wardle,2014,6.63
1,Dorothy Wardle,2015,-371.25
2,Dorothy Wardle,2016,92.61
3,Dorothy Wardle,2017,5.11
4,=--Katharine Harms,2014,261.33
...,...,...,...
2472,Zuschuss Carroll,2016,-1731.01
2473,Zuschuss Carroll,2017,62.34
2474,Zuschuss Donatelli,2014,25.49
2475,Zuschuss Donatelli,2016,206.67
